In [ ]:
pip install -r "requirements.txt"

# Import dependencies

In [1]:
import pandas as pd
import numpy as np

# Data

## Load resale data

In [5]:
resale = pd.read_csv("Data/Resale_with_Coords.csv")

In [6]:
print(f"Shape: {resale.shape}")
print(resale.info())
resale.describe(include='all').T

Shape: (225127, 14)
<class 'pandas.DataFrame'>
RangeIndex: 225127 entries, 0 to 225126
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                225127 non-null  str    
 1   town                 225127 non-null  str    
 2   flat_type            225127 non-null  str    
 3   block                225127 non-null  str    
 4   street_name          225127 non-null  str    
 5   storey_range         225127 non-null  str    
 6   floor_area_sqm       225127 non-null  float64
 7   flat_model           225127 non-null  str    
 8   lease_commence_date  225127 non-null  int64  
 9   remaining_lease      225127 non-null  str    
 10  resale_price         225127 non-null  float64
 11  address              225127 non-null  str    
 12  latitude             225127 non-null  float64
 13  longitude            225127 non-null  float64
dtypes: float64(4), int64(1), str(9)
memory usage: 24.0 MB
None


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
month,225127,110,2024-07,3036,NaN,NaN,NaN,NaN,NaN,NaN,NaN
town,225127,26,SENGKANG,18391,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flat_type,225127,7,4 ROOM,95464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
block,225127,2749,2,682,NaN,NaN,NaN,NaN,NaN,NaN,NaN
street_name,225127,577,YISHUN RING RD,3208,NaN,NaN,NaN,NaN,NaN,NaN,NaN
storey_range,225127,17,04 TO 06,51639,NaN,NaN,NaN,NaN,NaN,NaN,NaN
floor_area_sqm,225127.0,NaN,NaN,NaN,96.751932,24.019493,31.0,81.0,93.0,112.0,366.7
flat_model,225127,21,Model A,80573,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lease_commence_date,225127.0,NaN,NaN,NaN,1996.463969,14.314362,1966.0,1985.0,1997.0,2012.0,2021.0
remaining_lease,225127,696,94 years 10 months,1919,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Data Check [None]

In [ ]:
pd.DataFrame({
    'nunique': resale.nunique(dropna=False),
    'missing': resale.isna().sum(),
}).sort_values('nunique', ascending=False)

In [ ]:
resale=resale.dropna()

## Data Description

In [ ]:
# 'role': 'feature', 'target', 'identifier', 'metadata', 'ambiguous'
# 'type':  'num-cont', 'num-disc', 'cat-nom', 'cat-ord', 'bool'
role_map = {
    "month" : "feature",
    "town": "feature",
    "flat_type": "feature",
    "block": "identifier",
    "street_name": "identifier",
    "storey_range": "feature",
    "floor_area_sqm": "feature",
    "flat_model": "metadata",
    "lease_commence_date": "metadata",
    "remaining_lease": "feature",
    "resale_price": "target"
}
type_map = {
    "month" : "num-disc",
    "town": "cat-nom",
    "flat_type": "cat-nom",
    "block": "cat-nom",
    "street_name": "cat-nom",
    "storey_range": "cat-ord",
    "floor_area_sqm": "num-disc",
    "flat_model": "cat-nom",
    "lease_commence_date": "num-disc",
    "remaining_lease": "num-disc",
    "resale_price": "num-disc"
}

data_description = pd.DataFrame({
    "column": resale.columns,
})
data_description["role"] = data_description["column"].map(role_map).fillna("unknown")
data_description["type"] = data_description["column"].map(type_map).fillna("unknown")
data_description

## Data Pre-Processing

### Model Dataset Descriptions

In [ ]:
model_data_desc = data_description[data_description['role'] != "metadata"]
model_data_desc

### Model Dataset

In [ ]:
model_data = resale[model_data_desc["column"]].copy()
model_data.head()

### Data Transformation

**Month** \
Convert to indexing for model input, index represents the order for time-series models \
index starts from 2017-01 onwards \
Example:

| Month(before) | index(after) |
| ------ | ------ |
| 2017-01 | 0 |
| 2017-02 | 1 |

In [ ]:
months = pd.to_datetime(model_data['month'], format='%Y-%m')
model_data['month'] = (
    months.dt.year * 12 + months.dt.month
)

model_data['month'] -= model_data['month'].min()
model_data.head()

**remaining_lease [Months]** \
Convert remaining_lease to number of months instead of X years X months

In [ ]:
remaining_yrs = model_data['remaining_lease'].str.extract(r'(\d+)\D+(?:(\d+)\D+)?').fillna(0).astype(int)
model_data['remaining_lease'] = remaining_yrs[0] * 12 + remaining_yrs[1]
model_data.head()

**Storey type** \
Replace storey range with the storey type (lower, middle, upper)

In [ ]:
avg_storey = model_data['storey_range'].str.split(" TO ", expand=True).astype(int).mean(axis=1)
model_data['storey_type'] = pd.cut(avg_storey, bins=[1,3, 7,99], labels=['lower','middle','upper'], right=False)
model_data = pd.get_dummies(model_data, columns=['storey_type'])
model_data.drop(columns=['storey_range'], inplace=True)
model_data.head()

**Quarter index** \
Defines which quarter is the data in example 2 for 2017-04 to 2017-07

In [ ]:
model_data['quarter'] = model_data['month'] // 3
model_data.tail()

**Resale Price Index [RPI]**

In [ ]:
rpi = pd.read_csv('Data/2025-RPI.csv')
rpi = rpi[rpi['year'] >= 2017]
rpi['quarter'] = (rpi['year'] - 2017) * 4 + (rpi['quarter'] - 1)
rpi.drop(columns=['year'], inplace=True)
rpi.head()

In [ ]:
model_data = pd.merge(model_data, rpi, on='quarter',how='left')
model_data.head()

**Adjusted Resale Price** \
Formula: Resale price / RPI

In [ ]:
model_data['adjusted_resale'] = model_data['resale_price'] / (model_data['rpi'] / 100)
model_data.head()

**Remove unneccessary columns**

In [ ]:
transformed = model_data.drop(columns=['block','street_name','quarter','address'])
transformed.head()

# Feature Extraction

Add amenity count for the varies amenities (mrt,bus-stop, schools, etc..)

## Import Dependencies

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

## Load Data

In [ ]:
mrt_df = pd.read_csv("Data/mrt_stations.csv")
schools_df = pd.read_csv("Data/schools.csv")
malls_df = pd.read_csv("Data/shopping_malls.csv")
bus_stop_df = pd.read_csv("Data/bus_stops.csv")
hawker_df = pd.read_csv("Data/hawker_centres.csv")
wet_market_df = pd.read_csv("Data/wet_markets.csv")

## Project Data on Point

In [ ]:
mrt_gdf = gpd.GeoDataFrame(mrt_df, geometry=gpd.points_from_xy(mrt_df.long, mrt_df.lat), crs="EPSG:4326")
schools_gdf = gpd.GeoDataFrame(schools_df, geometry=gpd.points_from_xy(schools_df.long, schools_df.lat), crs="EPSG:4326")
malls_gdf = gpd.GeoDataFrame(malls_df, geometry=gpd.points_from_xy(malls_df.long, malls_df.lat), crs="EPSG:4326")
bus_stop_gdf = gpd.GeoDataFrame(bus_stop_df, geometry=gpd.points_from_xy(bus_stop_df.long, bus_stop_df.lat), crs="EPSG:4326")
hawker_gdf = gpd.GeoDataFrame(hawker_df, geometry=gpd.points_from_xy(hawker_df.long, hawker_df.lat), crs="EPSG:4326")
wet_market_gdf = gpd.GeoDataFrame(hawker_df, geometry=gpd.points_from_xy(hawker_df.long, hawker_df.lat), crs="EPSG:4326")

In [ ]:
flat_gdf = gpd.GeoDataFrame(transformed, geometry=gpd.points_from_xy(transformed.longitude, transformed.latitude), crs="EPSG:4326")
flat_gdf.head()

In [ ]:
# Reproject to a Metric CRS (Singapore SVY21 is EPSG:3414)
flat_gdf = flat_gdf.to_crs(epsg=3414)
mrt_gdf = mrt_gdf.to_crs(epsg=3414)
schools_gdf = schools_gdf.to_crs(epsg=3414)
malls_gdf = malls_gdf.to_crs(epsg=3414)
bus_stop_gdf = bus_stop_gdf.to_crs(epsg=3414)
hawker_gdf = hawker_gdf.to_crs(epsg=3414)
wet_market_gdf = wet_market_gdf.to_crs(epsg=3414)

## Create Buffer zone

In [ ]:
# Create a 500m buffer around each house
flat_gdf['geometry'] = flat_gdf.geometry.buffer(500)

## Get Nearby amenties in radius (500M) (New Feature columns)

In [ ]:
# Spatial Join: Count MRT stations inside the house buffers
joined = gpd.sjoin(flat_gdf, mrt_gdf, how="left", predicate="intersects")
school_joined = gpd.sjoin(flat_gdf, schools_gdf, how="left", predicate="intersects")
mall_joined = gpd.sjoin(flat_gdf, malls_gdf, how="left", predicate="intersects")
bus_stop_joined = gpd.sjoin(flat_gdf, bus_stop_gdf, how="left", predicate="intersects")
hawker_joined = gpd.sjoin(flat_gdf, hawker_gdf, how="left", predicate="intersects")
wet_market_joined = gpd.sjoin(flat_gdf, wet_market_gdf, how="left", predicate="intersects")

mrt_counts = joined.groupby(joined.index).size() - joined['index_right'].isna().groupby(joined.index).sum()
school_counts = school_joined.groupby(school_joined.index).size() - school_joined['index_right'].isna().groupby(school_joined.index).sum()
mall_counts = mall_joined.groupby(mall_joined.index).size() - mall_joined['index_right'].isna().groupby(mall_joined.index).sum()
bus_stop_counts = bus_stop_joined.groupby(bus_stop_joined.index).size() - bus_stop_joined['index_right'].isna().groupby(bus_stop_joined.index).sum()
hawker_counts = hawker_joined.groupby(hawker_joined.index).size() - hawker_joined['index_right'].isna().groupby(hawker_joined.index).sum()
wet_market_counts = wet_market_joined.groupby(wet_market_joined.index).size() - wet_market_joined['index_right'].isna().groupby(wet_market_joined.index).sum()

transformed['mrt_count'] = mrt_counts.values
transformed['school_count'] = school_counts.values
transformed['mall_count'] = mall_counts.values
transformed['bus_stop_count'] = bus_stop_counts.values
transformed['hawker_count'] = hawker_counts.values
transformed['wet_market_count'] = wet_market_counts.values
transformed.head()

## Calculate distance to nearest bus-stop and MRT station in meters (New Feature Columns)

In [ ]:
# Create GeoDataFrame from transformed data
flat_point_gdf = gpd.GeoDataFrame(
    transformed, 
    geometry=gpd.points_from_xy(transformed.longitude, transformed.latitude), 
    crs="EPSG:4326"
)
flat_point_gdf = flat_point_gdf.to_crs(epsg=3414)
print("✓ flat_point_gdf created and reprojected.")

# --- Calculate Distance to Nearest MRT ---
print("Calculating nearest MRT distances...")
mrt_distances = []

for idx, flat in flat_point_gdf.iterrows():
    distances_to_all_mrt = mrt_gdf.geometry.distance(flat.geometry)
    min_distance = distances_to_all_mrt.min()
    mrt_distances.append(min_distance)

transformed['nearest_mrt_dist'] = pd.Series(mrt_distances).round(0).astype(int)
print(f"✓ Nearest MRT distance calculated for {len(mrt_distances)} flats.")

# --- Calculate Distance to Nearest Bus Stop ---
print("Calculating nearest bus stop distances...")
bus_distances = []

for idx, flat in flat_point_gdf.iterrows():
    distances_to_all_bus = bus_stop_gdf.geometry.distance(flat.geometry)
    min_distance = distances_to_all_bus.min()
    bus_distances.append(min_distance)

transformed['nearest_bus_stop_dist'] = pd.Series(bus_distances).round(0).astype(int)
print(f"✓ Nearest bus stop distance calculated for {len(bus_distances)} flats.")

# --- Validation ---
print(f"\nValidation:")
print(f"  MRT - NaN values: {transformed['nearest_mrt_dist'].isna().sum()}")
print(f"  Bus - NaN values: {transformed['nearest_bus_stop_dist'].isna().sum()}")
print(f"  MRT - Mean distance: {transformed['nearest_mrt_dist'].mean():.0f}m")
print(f"  Bus - Mean distance: {transformed['nearest_bus_stop_dist'].mean():.0f}m")

display(transformed.head())

In [ ]:
import numpy as np

transformed['nearest_mrt_dist'] = np.floor(transformed['nearest_mrt_dist'])
transformed['nearest_bus_stop_dist'] = np.floor(transformed['nearest_bus_stop_dist'])

display(transformed.head())

The `nearest_mrt_dist` and `nearest_bus_stop_dist` columns have now been rounded down to the nearest whole number. You can see the updated values in the displayed DataFrame.

# Export Data

In [ ]:
transformed.to_csv('Data/processed.csv', index=False)

# Importing Data

### Data Understanding

In [9]:
resale = pd.read_csv("Data/processed.csv")
resale_extreme = resale[resale['floor_area_sqm'] > 350]  # drop the extreme outlier

summary = resale.groupby('flat_type').agg(
    Avg_Floor_Area=('floor_area_sqm', 'mean'),
    Avg_Resale_Price=('resale_price', 'mean'),
    Avg_Adjusted_Resale=('adjusted_resale', 'mean'),
    Count=('resale_price', 'count')
).round(2).sort_values('Avg_Floor_Area')

summary['Avg_Resale_Price'] = summary['Avg_Resale_Price'].apply(lambda x: f'${x:,.0f}')
summary['Avg_Adjusted_Resale'] = summary['Avg_Adjusted_Resale'].apply(lambda x: f'${x:,.0f}')
summary['Avg_Floor_Area'] = summary['Avg_Floor_Area'].apply(lambda x: f'{x:.1f} sqm')
display(summary)

,Avg_Floor_Area,Avg_Resale_Price,Avg_Adjusted_Resale,Count
flat_type,,,,
1 ROOM,31.0 sqm,"$211,987","$137,058",82
2 ROOM,45.7 sqm,"$301,466","$179,074",4586
3 ROOM,68.2 sqm,"$373,075","$233,229",53591
4 ROOM,95.0 sqm,"$530,607","$331,236",95464
5 ROOM,117.7 sqm,"$625,558","$395,549",55179
EXECUTIVE,144.8 sqm,"$733,328","$469,462",16140
MULTI-GENERATION,161.2 sqm,"$859,598","$576,238",85


In [ ]:
resale_extreme

## Feature Heatmap

## Feature Correlation observations (compared to adjusted resale)

## Outlier Analysis

In [ ]:
display(resale[numeric_cols].describe().round(2))

### IQR outlier detection

In [ ]:
outlier_summary = []
for col in numeric_cols:
    Q1 = resale[col].quantile(0.25)
    Q3 = resale[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = resale[(resale[col] < lower) | (resale[col] > upper)]
    pct = len(outliers) / len(resale) * 100
    
    outlier_summary.append({
        'Feature': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lower Bound': round(lower, 2),
        'Upper Bound': round(upper, 2),
        'Outlier Count': len(outliers),
        'Outlier %': round(pct, 2)
    })
outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

In [ ]:
print("FEATURE ANALYSIS RESULTS & INTERPRETATION")
print("="*80)

# Define target variable
target_col = 'adjusted_resale' if 'adjusted_resale' in resale.columns else 'resale_price'

# Get correlations with target
numeric_cols = resale.select_dtypes(include=[np.number]).columns
target_correlations = resale[numeric_cols].corr()[target_col].sort_values(ascending=False)

# Categorize features by correlation strength and type
feature_categories = {
    'Strong Predictors (|r| > 0.5)': [],
    'Moderate Predictors (0.3 < |r| < 0.5)': [],
    'Physical Characteristics': [],
    'Location Features': [],
    'Amenity Features (Count)': [],
    'Amenity Features (Distance)': [],
    'Redundant Features (To Drop)': [],
    'Low Variance Features (To Drop)': []
}

# Categorize each feature
for feature, corr in target_correlations.items():
    if feature == target_col:
        continue
    
    abs_corr = abs(corr)
    
    # Check for redundancy
    if feature in ['resale_price', 'rpi']:
        feature_categories['Redundant Features (To Drop)'].append((feature, corr))
    elif feature == 'month':
        feature_categories['Low Variance Features (To Drop)'].append((feature, corr))
    elif feature == 'wet_market_count':
        feature_categories['Redundant Features (To Drop)'].append((feature, corr))
    
    # Strong predictors
    elif abs_corr > 0.5:
        feature_categories['Strong Predictors (|r| > 0.5)'].append((feature, corr))
    
    # Moderate predictors
    elif abs_corr >= 0.3:
        feature_categories['Moderate Predictors (0.3 < |r| < 0.5)'].append((feature, corr))
    
    # Physical characteristics
    elif feature in ['floor_area_sqm', 'remaining_lease', 'storey_type_lower', 'storey_type_middle', 'storey_type_upper']:
        feature_categories['Physical Characteristics'].append((feature, corr))
    
    # Location
    elif feature in ['latitude', 'longitude']:
        feature_categories['Location Features'].append((feature, corr))
    
    # Amenity counts
    elif 'count' in feature.lower():
        feature_categories['Amenity Features (Count)'].append((feature, corr))
    
    # Amenity distances
    elif 'dist' in feature.lower() or 'nearest' in feature.lower():
        feature_categories['Amenity Features (Distance)'].append((feature, corr))

# Display categorized features
print("\n" + "="*80)
print("FEATURES CATEGORIZED BY TYPE AND CORRELATION STRENGTH")
print("="*80)

for category, features in feature_categories.items():
    if features:
        print(f"\n{'─'*80}")
        print(f"{category}:")
        print(f"{'─'*80}")
        for feature, corr in features:
            print(f"  {feature:.<40} r = {corr:>6.3f}")

### Results Discussion

#### Columns to keep

##### Floor_area_sqm, Remaining_lease, latitude, Longitude, All amenities.

Floor_area_sqm and remaining_lease are the biggest impact based on the features and have been cross checked with experts in the field.

Lat,long itself might not seem like much but typically locations closer to the center of singapore would be priced higher, hence would be a factor

Although amenities itself has low correlation to the adjusted resale, they do have a real impact when asking experts in the field. It was suggested that the prices can vary by about 10% based on the amenities itself. While alone they might not amount to much together they might have a compounding effect which is why it is not wise to drop them.

#### Columns to drop

##### resale price, wet_market_count, rpi, month

resale price would be data leakage.
month is very similar to remaining lease, hence will be dropped, also because low correlation.
wet_market_count is too similar to hawker centers, unless we fix the 1:1 correlation, we drop one of them.
rpi is unnecessary for model training until the final multiplicative step.

## Scatterplot

## Feature Heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Load processed data
resale = pd.read_csv("Data/processed.csv")

print(resale.columns.tolist())

# Show first few rows
display(resale.head())

# Quick data info
print("\n" + "="*80)
print("DATA INFO")
print("="*80)
print(resale.info())


In [ ]:
# Get numeric columns only
numeric_cols = resale.select_dtypes(include=[np.number]).columns.tolist()
print(f"Analyzing {len(numeric_cols)} numeric features\n")

# Calculate correlation matrix
correlation_matrix = resale[numeric_cols].corr()

# Create correlation heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Correlation heatmap generated")

## Feature Correlation observations (compared to adjusted resale)

## Outlier Analysis

## Scatterplot

In [ ]:
# ------------------------------------------------------------
# IQR WITHIN GROUPS (by flat_type)
# ------------------------------------------------------------

plot_df = resale.dropna(subset=['adjusted_resale']).copy()

def flag_outliers_iqr(group, col, multiplier=1.5):
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return (group[col] < lower) | (group[col] > upper)

# Flag outliers within each flat_type group
plot_df['is_outlier'] = (
    plot_df.groupby('flat_type', group_keys=False)
    .apply(lambda g: flag_outliers_iqr(g, 'adjusted_resale'))
)

# Summary table
print("=" * 60)
print("OUTLIER COUNT BY FLAT TYPE")
print("=" * 60)
summary = plot_df.groupby('flat_type')['is_outlier'].agg(
    Total='count',
    Outliers='sum'
)
summary['Outlier %'] = (summary['Outliers'] / summary['Total'] * 100).round(2)
display(summary)

print(f"\nTotal outliers: {plot_df['is_outlier'].sum():,} / {len(plot_df):,} ({plot_df['is_outlier'].mean()*100:.2f}%)")

# Scatter plot
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(plot_df.loc[~plot_df['is_outlier'], 'floor_area_sqm'],
           plot_df.loc[~plot_df['is_outlier'], 'adjusted_resale'],
           alpha=0.2, s=5, color='steelblue', label='Normal')

ax.scatter(plot_df.loc[plot_df['is_outlier'], 'floor_area_sqm'],
           plot_df.loc[plot_df['is_outlier'], 'adjusted_resale'],
           alpha=0.6, s=15, color='red', label='Outliers')

ax.set_xlabel('Floor Area (sqm)', fontsize=12)
ax.set_ylabel('Adjusted Resale Price (SGD)', fontsize=12)
ax.set_title('Adjusted Resale Price vs Floor Area\nOutliers Detected via Grouped IQR (by Flat Type)', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
display(resale[resale['floor_area_sqm'] >= 350])

### Outlier discussion

We detect outliers by room types instead of one whole column because if we do by whole column, any price above a certain sqm will be flagged as outliers. For this graph we use interquartile range (25-75%) where anything above and below is outlier.
Here we can see it is more correct where certain extreme ends of data points are flagged out.
However, since these are actual datapoints. I suggest we only remove the extreme outliers. (E.g. from the graph above the one above 350 sqm)

In [ ]:
display(resale[numeric_cols].describe().round(2))

### IQR outlier detection

In [ ]:
outlier_summary = []
for col in numeric_cols:
    Q1 = resale[col].quantile(0.25)
    Q3 = resale[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = resale[(resale[col] < lower) | (resale[col] > upper)]
    pct = len(outliers) / len(resale) * 100
    
    outlier_summary.append({
        'Feature': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lower Bound': round(lower, 2),
        'Upper Bound': round(upper, 2),
        'Outlier Count': len(outliers),
        'Outlier %': round(pct, 2)
    })
outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

In [ ]:
print("FEATURE ANALYSIS RESULTS & INTERPRETATION")
print("="*80)

# Define target variable
target_col = 'adjusted_resale' if 'adjusted_resale' in resale.columns else 'resale_price'

# Get correlations with target
numeric_cols = resale.select_dtypes(include=[np.number]).columns
target_correlations = resale[numeric_cols].corr()[target_col].sort_values(ascending=False)

# Categorize features by correlation strength and type
feature_categories = {
    'Strong Predictors (|r| > 0.5)': [],
    'Moderate Predictors (0.3 < |r| < 0.5)': [],
    'Physical Characteristics': [],
    'Location Features': [],
    'Amenity Features (Count)': [],
    'Amenity Features (Distance)': [],
    'Redundant Features (To Drop)': [],
    'Low Variance Features (To Drop)': []
}

# Categorize each feature
for feature, corr in target_correlations.items():
    if feature == target_col:
        continue
    
    abs_corr = abs(corr)
    
    # Check for redundancy
    if feature in ['resale_price', 'rpi']:
        feature_categories['Redundant Features (To Drop)'].append((feature, corr))
    elif feature == 'month':
        feature_categories['Low Variance Features (To Drop)'].append((feature, corr))
    elif feature == 'wet_market_count':
        feature_categories['Redundant Features (To Drop)'].append((feature, corr))
    
    # Strong predictors
    elif abs_corr > 0.5:
        feature_categories['Strong Predictors (|r| > 0.5)'].append((feature, corr))
    
    # Moderate predictors
    elif abs_corr >= 0.3:
        feature_categories['Moderate Predictors (0.3 < |r| < 0.5)'].append((feature, corr))
    
    # Physical characteristics
    elif feature in ['floor_area_sqm', 'remaining_lease', 'storey_type_lower', 'storey_type_middle', 'storey_type_upper']:
        feature_categories['Physical Characteristics'].append((feature, corr))
    
    # Location
    elif feature in ['latitude', 'longitude']:
        feature_categories['Location Features'].append((feature, corr))
    
    # Amenity counts
    elif 'count' in feature.lower():
        feature_categories['Amenity Features (Count)'].append((feature, corr))
    
    # Amenity distances
    elif 'dist' in feature.lower() or 'nearest' in feature.lower():
        feature_categories['Amenity Features (Distance)'].append((feature, corr))

# Display categorized features
print("\n" + "="*80)
print("FEATURES CATEGORIZED BY TYPE AND CORRELATION STRENGTH")
print("="*80)

for category, features in feature_categories.items():
    if features:
        print(f"\n{'─'*80}")
        print(f"{category}:")
        print(f"{'─'*80}")
        for feature, corr in features:
            print(f"  {feature:.<40} r = {corr:>6.3f}")

### Results Discussion

#### Columns to keep

##### Floor_area_sqm, Remaining_lease, latitude, Longitude, All amenities.

Floor_area_sqm and remaining_lease are the biggest impact based on the features and have been cross checked with experts in the field.

Lat,long itself might not seem like much but typically locations closer to the center of singapore would be priced higher, hence would be a factor

Although amenities itself has low correlation to the adjusted resale, they do have a real impact when asking experts in the field. It was suggested that the prices can vary by about 10% based on the amenities itself. While alone they might not amount to much together they might have a compounding effect which is why it is not wise to drop them.

#### Columns to drop

##### resale price, wet_market_count, rpi, month

resale price would be data leakage.
month is very similar to remaining lease, hence will be dropped, also because low correlation.
wet_market_count is too similar to hawker centers, unless we fix the 1:1 correlation, we drop one of them.
rpi is unnecessary for model training until the final multiplicative step.

In [ ]:
# ------------------------------------------------------------
# IQR WITHIN GROUPS (by flat_type)
# ------------------------------------------------------------

plot_df = resale.dropna(subset=['adjusted_resale']).copy()

def flag_outliers_iqr(group, col, multiplier=1.5):
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return (group[col] < lower) | (group[col] > upper)

# Flag outliers within each flat_type group
plot_df['is_outlier'] = (
    plot_df.groupby('flat_type', group_keys=False)
    .apply(lambda g: flag_outliers_iqr(g, 'adjusted_resale'))
)

# Summary table
print("=" * 60)
print("OUTLIER COUNT BY FLAT TYPE")
print("=" * 60)
summary = plot_df.groupby('flat_type')['is_outlier'].agg(
    Total='count',
    Outliers='sum'
)
summary['Outlier %'] = (summary['Outliers'] / summary['Total'] * 100).round(2)
display(summary)

print(f"\nTotal outliers: {plot_df['is_outlier'].sum():,} / {len(plot_df):,} ({plot_df['is_outlier'].mean()*100:.2f}%)")

# Scatter plot
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(plot_df.loc[~plot_df['is_outlier'], 'floor_area_sqm'],
           plot_df.loc[~plot_df['is_outlier'], 'adjusted_resale'],
           alpha=0.2, s=5, color='steelblue', label='Normal')

ax.scatter(plot_df.loc[plot_df['is_outlier'], 'floor_area_sqm'],
           plot_df.loc[plot_df['is_outlier'], 'adjusted_resale'],
           alpha=0.6, s=15, color='red', label='Outliers')

ax.set_xlabel('Floor Area (sqm)', fontsize=12)
ax.set_ylabel('Adjusted Resale Price (SGD)', fontsize=12)
ax.set_title('Adjusted Resale Price vs Floor Area\nOutliers Detected via Grouped IQR (by Flat Type)', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
display(resale[resale['floor_area_sqm'] >= 350])

### Outlier discussion

We detect outliers by room types instead of one whole column because if we do by whole column, any price above a certain sqm will be flagged as outliers. For this graph we use interquartile range (25-75%) where anything above and below is outlier.
Here we can see it is more correct where certain extreme ends of data points are flagged out.
However, since these are actual datapoints. I suggest we only remove the extreme outliers. (E.g. from the graph above the one above 350 sqm)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Load processed data
resale = pd.read_csv("Data/processed.csv")

print(resale.columns.tolist())

# Show first few rows
display(resale.head())

# Quick data info
print("\n" + "="*80)
print("DATA INFO")
print("="*80)
print(resale.info())


In [ ]:
# Get numeric columns only
numeric_cols = resale.select_dtypes(include=[np.number]).columns.tolist()
print(f"Analyzing {len(numeric_cols)} numeric features\n")

# Calculate correlation matrix
correlation_matrix = resale[numeric_cols].corr()

# Create correlation heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Correlation heatmap generated")

# Price Prediction Models

## Load data

In [2]:
import pandas as pd

resale = pd.read_csv("Data/processed.csv")
resale.tail()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,...,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist
224537,108,YISHUN,EXECUTIVE,146.0,743,920000.0,1.429239,103.842146,False,True,...,203.4,452310.717797,0,3,1,14,0,0,792,180
224538,108,YISHUN,EXECUTIVE,142.0,739,865888.0,1.427737,103.845686,False,False,...,203.4,425706.981318,0,3,0,17,1,1,1200,88
224539,108,YISHUN,EXECUTIVE,142.0,729,825000.0,1.421335,103.837437,False,False,...,203.4,405604.719764,0,3,0,19,0,0,660,115
224540,108,YISHUN,EXECUTIVE,146.0,728,788000.0,1.421335,103.837437,False,True,...,203.4,387413.962635,0,3,0,19,0,0,660,115
224541,109,YISHUN,EXECUTIVE,146.0,730,860088.0,1.420201,103.836153,False,True,...,203.4,422855.457227,1,2,0,22,0,0,469,138


In [3]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Separate features and target
resale = resale.dropna(subset=["nearest_mrt_dist", "nearest_bus_stop_dist"])
X = resale.drop(columns=['floor_area_sqm', 'adjusted_resale','resale_price', 'latitude', 'longitude', 'month', 'rpi'])
y = resale["adjusted_resale"]

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create 5 bins based on price distribution  (low, medium, high for 3 bins)
y_binned = pd.qcut(y, q=5, labels=False)

# First split: Train (60%) and Temp (40%)
X_train, X_temp, y_train, y_temp, yb_train, yb_temp = train_test_split(
    X, y, y_binned,
    test_size=0.4,
    stratify=y_binned,
    random_state=42
)

# Second split: Validation (20%) and Test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=yb_temp,
    random_state=42
)


# Check sizes
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (134725, 43)
Validation set: (44908, 43)
Test set: (44909, 43)


## Dummy Regressor

In [4]:
# Initialize dummy model (predicts mean of y_train)
dummy_model = DummyRegressor(strategy="mean")

# Train
dummy_model.fit(X_train, y_train)

# Predict on test set
dummy_pred = dummy_model.predict(X_test)

# Evaluate
print("Dummy Baseline Pelrormance:")
print("MAE:", mean_absolute_error(y_test, dummy_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dummy_pred)))
print("R2:", r2_score(y_test, dummy_pred))

Dummy Baseline Pelrormance:
MAE: 83094.00326267991
RMSE: 108116.49177632012
R2: -2.7113559619706962e-08


## Linear Regression

In [5]:
# Initialize model
lr_model = LinearRegression()

# Train
lr_model.fit(X_train, y_train)

# Predict
lr_pred = lr_model.predict(X_test)

# Evaluate
print("Linear Regression Pelrormance:")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("R2:", r2_score(y_test, lr_pred))

# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_r2, lr_mae, lr_rmse = [], [], []

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    lr_cv = LinearRegression()
    lr_cv.fit(X_tr, y_tr)
    pred = lr_cv.predict(X_val)

    lr_r2.append(r2_score(y_val, pred))
    lr_mae.append(mean_absolute_error(y_val, pred))
    lr_rmse.append(np.sqrt(mean_squared_error(y_val, pred)))

print("\nLinear Regression Cross Validation Results")
print("Mean R2:", np.mean(lr_r2))
print("Std R2:", np.std(lr_r2))
print("Mean MAE:", np.mean(lr_mae))
print("Mean RMSE:", np.mean(lr_rmse))

Linear Regression Pelrormance:
MAE: 34453.130705412004
RMSE: 45685.20993000975
R2: 0.8214469095411567

Linear Regression Cross Validation Results
Mean R2: 0.8192416581152486
Std R2: 0.0031254647999313526
Mean MAE: 34505.47406150297
Mean RMSE: 45803.70452114834


## Random Forest

In [6]:
# Init the RF model
rf_model = RandomForestRegressor(random_state=1)

# Train
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
print("Random Forest Performance:")
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("R2:", r2_score(y_test, rf_pred))

# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_r2, rf_mae, rf_rmse = [], [], []

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    rf_cv = RandomForestRegressor(random_state=42, n_jobs=-1)
    rf_cv.fit(X_tr, y_tr)
    pred = rf_cv.predict(X_val)

    rf_r2.append(r2_score(y_val, pred))
    rf_mae.append(mean_absolute_error(y_val, pred))
    rf_rmse.append(np.sqrt(mean_squared_error(y_val, pred)))

print("\nRandom Forest Cross Validation Results")
print("Mean R2:", np.mean(rf_r2))
print("Std R2:", np.std(rf_r2))
print("Mean MAE:", np.mean(rf_mae))
print("Mean RMSE:", np.mean(rf_rmse))

Random Forest Performance:
MAE: 15355.106314886636
RMSE: 22258.884162303155
R2: 0.9576139544553719

Random Forest Cross Validation Results
Mean R2: 0.9552915990688768
Std R2: 0.0009993733119397037
Mean MAE: 15697.160038379854
Mean RMSE: 22778.24479226116


# Controlled Ablations & Tuning

# RPI

## RPI Calculation

In [8]:
rpi = pd.read_csv('Data/2025-RPI.csv')
avg_rpi = rpi['rpi'].diff().mean()
avg_rpi = round(avg_rpi,3)

BASE_RPI = 133.9 # Base RPI as of 2017-01
print(avg_rpi)

1.521


## Metrics with RPI added in

### Preparing the data

In [9]:
y_actual = resale["resale_price"]
X = resale.drop(columns=['floor_area_sqm', 'adjusted_resale','resale_price', 'latitude', 'longitude','rpi'])

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# Create 5 bins based on price distribution  (low, medium, high for 3 bins)
y_binned = pd.qcut(y, q=5, labels=False)

# First split: Train (60%) and Temp (40%)
X_act_train, X_act_temp, y_act_train, y_act_temp, yb_train, yb_temp = train_test_split(
    X, y_actual, y_binned,
    test_size=0.4,
    stratify=y_binned,
    random_state=42
)

# Second split: Validation (20%) and Test (20%)
X_act_val, X_act_test, y_act_val, y_act_test = train_test_split(
    X_act_temp, y_act_temp,
    test_size=0.5,
    stratify=yb_temp,
    random_state=42
)

month_x = X_act_test[['month']].copy()
X_act_test = X_act_test.drop(columns=['month'])

### Linear Regression

In [10]:
# Get the predicated value using the model already trained above
lr_test = X_act_test.copy()
lr_pred = lr_model.predict(lr_test)

lr_rpi_test = pd.DataFrame()
lr_rpi_test.index = lr_test.index.copy()
lr_rpi_test["Pred"] = lr_pred

months = month_x.copy()
lr_rpi_test["month"] = months
lr_rpi_test["Pred_Actual"] = lr_rpi_test['Pred'] * ((lr_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

print("Logistic Regression RPI Result\n", lr_rpi_test.tail(), "\n")

Logistic Regression RPI Result
                  Pred  month    Pred_Actual
59853   175130.854366     33  263801.357241
139618  235288.905757     64  390205.474196
118888  396240.514830     66  663156.050429
222937  350301.097792    108  660864.039051
4148    248637.442588      3  336707.311127 



In [11]:
# Evaluate
print("Linear Regression Performance:")
print("MAE:", mean_absolute_error(y_act_test, lr_rpi_test["Pred_Actual"]))
print("RMSE:", np.sqrt(mean_squared_error(y_act_test, lr_rpi_test["Pred_Actual"])))
print("R2:", r2_score(y_act_test, lr_rpi_test["Pred_Actual"]))

Linear Regression Performance:
MAE: 66403.7700105703
RMSE: 84495.49536623403
R2: 0.7979370329881027


In [12]:
overshot = (y_act_test < lr_rpi_test['Pred_Actual']).sum()
undershot = (y_act_test > lr_rpi_test['Pred_Actual']).sum()

print(f"Overshot: {overshot} \nUndershot: {undershot}")

Overshot: 25814 
Undershot: 19095


#### Cross validation [WIP]

In [13]:
# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_r2, lr_mae, lr_rmse = [], [], []

for train_idx, val_idx in kf.split(X_act_train):
    X_tr, X_val = X_act_train.iloc[train_idx], X_act_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_act_train.iloc[val_idx]

    lr_cv = LinearRegression()
    months = X_val['month'].copy()
    X_tr = X_tr.drop(columns=['month'])
    X_val = X_val.drop(columns=['month'])
    lr_cv.fit(X_tr, y_tr)
    pred = lr_cv.predict(X_val)
    
    lr_rpi_test = pd.DataFrame()
    lr_rpi_test.index = X_val.index.copy()
    lr_rpi_test["Pred"] = pred
    lr_rpi_test["month"] = months
    pred_actual = lr_rpi_test['Pred'] * ((lr_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

    lr_r2.append(r2_score(y_val, pred_actual))
    lr_mae.append(mean_absolute_error(y_val, pred_actual))
    lr_rmse.append(np.sqrt(mean_squared_error(y_val, pred_actual)))

print("\nLinear Regression Cross Validation Results")
print("Mean R2:", np.mean(lr_r2))
print("Std R2:", np.std(lr_r2))
print("Mean MAE:", np.mean(lr_mae))
print("Mean RMSE:", np.mean(lr_rmse))


Linear Regression Cross Validation Results
Mean R2: 0.7959971300811949
Std R2: 0.0031572880297887147
Mean MAE: 66562.52409191996
Mean RMSE: 84584.39980761522


### Random Forest

In [14]:
rf_pred = rf_model.predict(X_act_test)

months = month_x.copy()

rf_rpi_test = pd.DataFrame()
rf_rpi_test.index = X_act_test.index.copy()
rf_rpi_test["Pred"] = rf_pred
rf_rpi_test["month"] = months
rf_rpi_test["Pred_Actual"] = rf_rpi_test['Pred'] *  ((rf_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

print("Random Forest RPI Result\n", rf_rpi_test.tail(), "\n")

Random Forest RPI Result
                  Pred  month    Pred_Actual
59853   183138.245870     33  275862.971136
139618  250834.848854     64  415987.021687
118888  336554.623782     66  563264.549454
222937  326223.882924    108  615440.928570
4148    239409.246481      3  324210.395678 



In [15]:
# Evaluate
print("Random Forest Performance:")
print("MAE:", mean_absolute_error(y_act_test, rf_rpi_test["Pred_Actual"]))
print("RMSE:", np.sqrt(mean_squared_error(y_act_test, rf_rpi_test["Pred_Actual"])))
print("R2:", r2_score(y_act_test, rf_rpi_test["Pred_Actual"]))

Random Forest Performance:
MAE: 43469.57355194791
RMSE: 55867.71392643529
R2: 0.9116632505859231


In [16]:
overshot = (y_act_test < rf_rpi_test['Pred_Actual']).sum()
undershot = (y_act_test > rf_rpi_test['Pred_Actual']).sum()

print(f"Overshot: {overshot} \nUndershot: {undershot}")

Overshot: 25965 
Undershot: 18944


#### Cross validation [WIP]

In [17]:
# 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_r2, rf_mae, rf_rmse = [], [], []

for train_idx, val_idx in kf.split(X_act_train):
    X_tr, X_val = X_act_train.iloc[train_idx], X_act_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_act_train.iloc[val_idx]

    rf_cv = RandomForestRegressor(random_state=42, n_jobs=-1)
    month_x = X_val['month'].copy()
    X_tr = X_tr.drop(columns=['month'])
    X_val = X_val.drop(columns=['month'])
    rf_cv.fit(X_tr, y_tr)
    pred = rf_cv.predict(X_val)
    
    rf_rpi_test = pd.DataFrame()
    rf_rpi_test.index = X_val.index.copy()
    rf_rpi_test["Pred"] = pred
    rf_rpi_test["month"] = month_x
    pred_actual = rf_rpi_test['Pred'] *  ((rf_rpi_test["month"] // 3 * avg_rpi + BASE_RPI) / 100)

    rf_r2.append(r2_score(y_val, pred_actual))
    rf_mae.append(mean_absolute_error(y_val, pred_actual))
    rf_rmse.append(np.sqrt(mean_squared_error(y_val, pred_actual)))

print("\nRandom Forest Cross Validation Results")
print("Mean R2:", np.mean(rf_r2))
print("Std R2:", np.std(rf_r2))
print("Mean MAE:", np.mean(rf_mae))
print("Mean RMSE:", np.mean(rf_rmse))


Random Forest Cross Validation Results
Mean R2: 0.9097778184313399
Std R2: 0.001498682623617223
Mean MAE: 43733.007752488615
Mean RMSE: 56251.046408711234
